# 

In [103]:
# Install required packages 
%pip install -U -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [104]:
import boto3
from datetime import datetime, date
from boto3.session import Session
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool

In [105]:
boto_session = Session()
REGION = boto_session.region_name
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

In [106]:
@tool
def extract_document_metadata(document_text: str) -> str:
    """Improved document type detection with better coverage."""
    text = document_text.lower()
    text_preview = document_text.strip()[:400]

    if any(kw in text for kw in ["basic life support", "bls provider", "bls card", "heartsaver"]):
        doc_type = "BLS"
    elif any(kw in text for kw in ["advanced cardiac life support", "acls provider", "acls card"]):
        doc_type = "ACLS"
    elif any(kw in text for kw in ["tuberculosis", "tb test", "ppd", "quantiferon", "t-spot", "tb screening"]):
        doc_type = "TB_TEST"
    elif ("resume" in text or "curriculum vitae" in text or 
          ("rn license" in text and len(document_text) > 250)):
        doc_type = "RESUME"
    elif any(kw in text for kw in ["competency", "assessment", "exam score", "/100", "skills checklist"]):
        doc_type = "ASSESSMENT"
    else:
        doc_type = "UNKNOWN"

    return f"""Document type detected: {doc_type}
Raw text length: {len(document_text)} characters
Preview: {text_preview}...
"""

In [107]:
@tool
def check_compliance_rules(doc_type: str, expiry_date: str = "", 
                           issuer: str = "", score: int = -1,
                           test_date: str = "") -> str:
    """
    Strict compliance rules engine.
    """
    today = date.today()
    results = [f"Document Type: {doc_type}"]

    issuer_lower = issuer.lower() if issuer else ""

    if doc_type == "BLS":
        # Expiry
        if expiry_date:
            try:
                exp = date.fromisoformat(expiry_date)
                results.append(f"Expiry check: {'PASS' if exp >= today else 'FAIL'} (expires {expiry_date})")
            except:
                results.append("Expiry check: FAIL — invalid expiry date format")
        else:
            results.append("Expiry check: FAIL — no expiry date found")

        # Issuer
        valid_issuers = ["american heart association", "aha", "american red cross", "red cross"]
        issuer_ok = any(v in issuer_lower for v in valid_issuers)
        results.append(f"Issuer check: {'PASS' if issuer_ok else 'FAIL'} (issuer: {issuer or 'not found'})")

    elif doc_type == "ACLS":
        # Expiry
        if expiry_date:
            try:
                exp = date.fromisoformat(expiry_date)
                results.append(f"Expiry check: {'PASS' if exp >= today else 'FAIL'} (expires {expiry_date})")
            except:
                results.append("Expiry check: FAIL — invalid expiry date format")
        else:
            results.append("Expiry check: FAIL — no expiry date found")

        # STRICT ACLS Issuer Rule — Only AHA
        if "american heart association" in issuer_lower or "aha" in issuer_lower:
            results.append("Issuer check: PASS (AHA — approved)")
        else:
            results.append(f"Issuer check: FAIL (issuer: {issuer or 'unknown'}) — ACLS must be issued by American Heart Association only")

        # Explicit non-AHA warning
        if any(bad in issuer_lower for bad in ["emedcert", "emed cert", "certifynow", "online", "national cpr"]):
            results.append("CRITICAL: Non-AHA provider detected — almost always rejected by hospitals")

    elif doc_type == "TB_TEST":
        if test_date:
            try:
                td = date.fromisoformat(test_date)
                days_elapsed = (today - td).days
                results.append(f"Recency check: {'PASS' if days_elapsed <= 365 else 'FAIL'} ({days_elapsed} days old)")
            except:
                results.append("Recency check: FAIL — invalid test date format")
        else:
            results.append("Recency check: FAIL — no test date found")

    elif doc_type == "RESUME":
        results.append("Resume review: Agent will verify active RN license + valid BLS + relevant experience")

    elif doc_type == "ASSESSMENT":
        if score >= 0:
            results.append(f"Score check: {'PASS' if score >= 80 else 'FAIL'} (score: {score}/100, minimum 80)")
        else:
            results.append("Score check: FAIL — no score found")

    else:
        results.append("UNKNOWN document type — manual review required")

    return "\n".join(results)

In [108]:
@tool
def flag_risk_patterns(document_text: str, doc_type: str) -> str:
    flags = []
    text_lower = document_text.lower()

    if doc_type in ["BLS", "ACLS"]:
        if "HIGH RISK" in document_text or any(x in text_lower for x in ["emedcert", "emed cert"]):
            flags.append("HIGH RISK: Non-AHA provider detected")
        elif "online only" in text_lower or "cognitive only" in text_lower:
            flags.append("WARNING: Online-only certification")

    if len(document_text.strip()) < 150:
        flags.append("WARNING: Document text unusually short")

    return "\n".join(flags) if flags else "No major risk flags"

In [109]:
@tool
def lookup_prior_decisions(doc_type: str, query: str) -> str:
    """
    Search AgentCore Memory for similar past compliance decisions.
    Args:
        doc_type: Document type to scope the search
        query: What to search for (e.g. 'expired BLS from Red Cross')
    Returns:
        Relevant past decisions retrieved from memory
    """
    return f"[Memory lookup for {doc_type}: '{query}'] — no prior decisions seeded yet. Will populate after Lab 2."

In [110]:
@tool
def extract_structured_fields(image_path: str) -> str:
    """
    Uses Claude Vision to analyze the document image directly.
    This can actually 'see' handwritten signatures, issuer logos, dates, etc.
    """
    try:
        from pathlib import Path
        import boto3, base64, json

        img_bytes = Path(image_path).read_bytes()
        mime = "image/png" if str(image_path).lower().endswith(".png") else "image/jpeg"
        b64 = base64.b64encode(img_bytes).decode()

        bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

        vision_prompt = """Look at this healthcare certification document image.

Your ONLY job: determine if a handwritten signature is present.

Answer these two questions only:
1. Is there a handwritten signature visible? Yes / No
2. If Yes — where is it located? (e.g. "bottom right, next to Instructor Name")

Return nothing else. Do not extract names, dates, or any other fields."""

        response = bedrock_runtime.invoke_model(
            modelId=VISION_MODEL,
            body=json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 1000,
                "temperature": 0,
                "messages": [{
                    "role": "user",
                    "content": [
                        {"type": "image", "source": {"type": "base64", "media_type": mime, "data": b64}},
                        {"type": "text", "text": vision_prompt}
                    ]
                }]
            })
        )

        result = json.loads(response['body'].read())['content'][0]['text']
        return f"VISION ANALYSIS:\n{result}"

    except Exception as e:
        return f"Error during vision analysis: {str(e)}"

In [111]:
SYSTEM_PROMPT = """You are a strict, professional healthcare compliance reviewer.

Consider date of checking as 28th April 2026

When vision analysis is available, prioritize it — especially for signature detection.

Key Rules:
- ACLS: ONLY American Heart Association (AHA) accepted. Any other issuer = REJECT.
- BLS: AHA or American Red Cross accepted.
- TB Test: Within last 365 days.
- Resume: Active nursing license + current BLS required.
- Assessment: Score >= 80/100.

Memory Usage Rules:
- Memory is for reference only
- Do NOT assume the current document is identical to past documents
- If similarities exist, say "similar case found" NOT "same document"
- Always prioritize current document evidence over memory
- If OCR and Vision conflict → ignore memory completely

After using tools, provide a short final decision.

You MUST end every response with exactly these three lines:

DECISION: APPROVE or REJECT
REASON: One short, clear sentence (max 15 words)
RULES CHECKED: Issuer, Expiry, Signature, etc. (e.g. Issuer: PASS, Expiry: PASS, Signature: PASS)

Be concise. No long explanations after the three lines.
"""

In [112]:
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

agent = Agent(
    model=model,
    tools=[extract_document_metadata, check_compliance_rules, 
           flag_risk_patterns, lookup_prior_decisions,extract_structured_fields],
    system_prompt=SYSTEM_PROMPT,
)

In [113]:
import uuid
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

REVIEWER_ID = "compliance-reviewer-001"

# ── Create a NEW Memory with valid name ─────────────────────────────────────
memory_manager = MemoryManager(region_name=REGION)

memory = memory_manager.get_or_create_memory(
    name="ComplianceMemory_v2",          # ← Fixed: underscore instead of hyphen
    strategies=[
        {
            StrategyType.SEMANTIC.value: {
                "name": "PastDecisions",
                "description": "Stores past compliance decisions and their rationale for BLS, ACLS, TB, Resume, and Assessments",
                "namespaces": ["compliance/decisions/{actorId}/semantic/"],
            }
        },
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "ReviewerPatterns",
                "description": "Captures reviewer strictness preferences and common rejection patterns",
                "namespaces": ["compliance/reviewer/{actorId}/preferences/"],
            }
        },
    ]
)

memory_id = memory["id"]
print(f"✅ New Memory ID: {memory_id}")

# ── Seed improved past decisions ───────────────────────────────────────────
memory_client = MemoryClient(region_name=REGION)

past_decisions = [
    ("BLS cert from American Red Cross, expires 2026-03-15, signed by instructor.", "USER"),
    ("APPROVED. Expiry valid, Red Cross is accredited issuer, signature field present.", "ASSISTANT"),

    ("ACLS cert issued by eMedCert, expiry 2027-01-01.", "USER"),
    ("REJECTED. ACLS must be issued by American Heart Association only. eMedCert is not acceptable.", "ASSISTANT"),

    ("ACLS cert from online-only provider 'CertifyNow'.", "USER"),
    ("REJECTED. AHA requires hands-on skills verification. Online-only ACLS not accepted.", "ASSISTANT"),

    ("TB test dated 14 months ago, result negative.", "USER"),
    ("REJECTED. TB test must be within last 12 months.", "ASSISTANT"),

    ("Resume with active New York RN license, current BLS, 3 years CVICU experience.", "USER"),
    ("APPROVED. Active license and valid BLS certification present with relevant experience.", "ASSISTANT"),

    ("Competency assessment score: 74/100.", "USER"),
    ("REJECTED. Minimum passing score is 80. Score of 74 does not meet threshold.", "ASSISTANT"),

    ("BLS card with 'Instructor Signature' field but OCR missed the actual signature.", "USER"),
    ("APPROVED. Signature field is present on official AHA card — acceptable.", "ASSISTANT"),
]

memory_client.create_event(
    memory_id=memory_id,
    actor_id=REVIEWER_ID,
    session_id="seed-session-001",
    messages=past_decisions,
)
print("✅ Seeded improved past decisions into new memory")

# ── Configure memory for the agent ─────────────────────────────────────────
memory_config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=str(uuid.uuid4()),
    actor_id=REVIEWER_ID,
    retrieval_config={
        "compliance/decisions/{actorId}/semantic/": RetrievalConfig(top_k=4, relevance_score=0.25),
        "compliance/reviewer/{actorId}/preferences/": RetrievalConfig(top_k=2, relevance_score=0.2),
    }
)

# Create the agent (update tools list as you add new ones)
agent = Agent(
    model=model,
    session_manager=AgentCoreMemorySessionManager(memory_config, REGION),
    tools=[
        extract_document_metadata, 
        check_compliance_rules,
        flag_risk_patterns, 
        lookup_prior_decisions,
        extract_structured_fields  
    ],
    system_prompt=SYSTEM_PROMPT,
)

✅ MemoryManager initialized for region: us-west-2
Memory already exists. Using existing memory ID: ComplianceMemory_v2-zPWzPA3FPK
🔎 Retrieving memory resource with ID: ComplianceMemory_v2-zPWzPA3FPK...
  Found memory: ComplianceMemory_v2-zPWzPA3FPK
Existing {'type': 'SEMANTIC', 'name': 'PastDecisions', 'description': 'Stores past compliance decisions and their rationale for BLS, ACLS, TB, Resume, and Assessments', 'namespaces': ['compliance/decisions/{actorId}/semantic/']}
Requested {'type': 'SEMANTIC', 'name': 'PastDecisions', 'description': 'Stores past compliance decisions and their rationale for BLS, ACLS, TB, Resume, and Assessments', 'namespaces': ['compliance/decisions/{actorId}/semantic/']}
Existing {'type': 'USER_PREFERENCE', 'name': 'ReviewerPatterns', 'description': 'Captures reviewer strictness preferences and common rejection patterns', 'namespaces': ['compliance/reviewer/{actorId}/preferences/']}
Requested {'type': 'USER_PREFERENCE', 'name': 'ReviewerPatterns', 'descripti

✅ New Memory ID: ComplianceMemory_v2-zPWzPA3FPK
✅ Seeded improved past decisions into new memory


In [128]:
import boto3, base64, json
from pathlib import Path

VISION_MODEL = "us.anthropic.claude-sonnet-4-20250514-v1:0"
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

def ocr_image(image_path: str) -> str:
    img_bytes = Path(image_path).read_bytes()
    mime = "image/png" if image_path.endswith(".png") else "image/jpeg"
    b64 = base64.b64encode(img_bytes).decode()

    response = bedrock_runtime.invoke_model(
        modelId=VISION_MODEL,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1000,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": mime, "data": b64}},
                    {"type": "text", "text": "Extract ALL text from this document image exactly as it appears. Return only the raw text, nothing else."}
                ]
            }]
        })
    )
    return json.loads(response['body'].read())['content'][0]['text']



IMAGE_PATH = "./images/Resume-01.png"   

print(f"Analyzing: {IMAGE_PATH}")
print("Running Vision Analysis...")

vision_result = extract_structured_fields(IMAGE_PATH)
print(vision_result)

extracted_text = ocr_image(IMAGE_PATH)

response = agent(vision_result + "\n\nRaw OCR text for reference:\n" + extracted_text)

print("\n" + "="*60)
print("FINAL DECISION")
print("="*60)
print(get_response_text(response))

Analyzing: ./images/Resume-02.png
Running Vision Analysis...
VISION ANALYSIS:
1. Is there a handwritten signature visible? No
2. If Yes — where is it located? N/A


I'll analyze this resume step by step.
Tool #54: extract_document_metadata

Tool #55: check_compliance_rules

Tool #56: lookup_prior_decisions

Tool #57: flag_risk_patterns
This resume is for a Certified Surgical Technologist position, but the compliance requirements mandate an active Registered Nurse (RN) license. Shawn White holds a Certified Surgical Technologist (CST) certification and has 8 years of relevant surgical experience, but lacks the required RN license. While the candidate has BLS certification, the issuer is not specified (only listed as "BLS | 2023" without indicating American Heart Association or American Red Cross), and most critically, there is no RN license present.

DECISION: REJECT
REASON: No RN license - position requires active nursing license
RULES CHECKED: License: FAIL, BLS: INCOMPLETE, Experienc